# Universal Context Engine (Converged Edition)

**Copyright 2026, Denis Rothman**

**Goal:** In this notebook we will integrate **Oracle 23ai (Converged Enterprise DB):** agents in a universal content engine to prove that the exact same reasoning architecture (engine.py) can seamlessly switch between:

*  Oracle 23ai (Vector Store): For Marketing & Legal use cases.
*  Oracle 23ai (Relational Table): For HR & Recruitment use cases.


# 1.Installation & Setup

In [36]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


In [37]:
# Install BOTH Pinecone and Oracle dependencies
!pip install oracledb==3.4.1 openai==2.14.0 tiktoken==0.7.0 tqdm==4.67.1 tenacity --quiet
print("✅ Universal Dependencies installed.")

✅ Universal Dependencies installed.


In [38]:
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [39]:
# Initialize Oracle Client

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

✅ OpenAI initialized (Mandatory).


In [40]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13119  100 13119    0     0  45713      0 --:--:-- --:--:-- --:--:-- 45870
Archive:  engine.zip
  inflating: /content/agent_archivist.py  
  inflating: /content/agent_recruiter.py  
  inflating: /content/agents.py      
  inflating: /content/engine.py      
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/registry.py    


In [41]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

✅ Universal Engine Modules imported.


In [42]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run Pinecone scenarios)")

✅ Oracle 23ai Connection established.


# 2.Initialize the Universal Engine

In [43]:
# 2. Initialize the Universal Engine Wrapper
from engine import context_engine

# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        # [FIX] Removed Pinecone Client (pc), Index, and Namespaces
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small"
    )

print("🚀 Universal Context Engine Wrapper is READY.")
# Access the registry dictionary keys directly to see the list of agents
print(list(AGENT_TOOLKIT.registry.keys()))

🚀 Universal Context Engine Wrapper is READY.
['Writer', 'Summarizer', 'OracleArchivist', 'OracleRecruiter']


# 3.Recruitment Scenarios
This execution plan switches to the `OracleRecruiter` agent. The Engine doesn't change code, only the configuration.

In [44]:
#@title Oracle data verification
from oracle_lib import OracleManager

print("\n=== Oracle Hybrid Table Summary ===\n")

# Use the Manager's context manager instead of a raw 'connection' variable
with OracleManager.get_cursor() as cursor:
    # --- 1. Check HR Tables ---
    tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

    for t in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {t}")
            result = cursor.fetchone()
            if result:
                print(f"Table {t}: {result[0]} rows")
        except Exception as e:
            print(f"Table {t}: Not Found or Error ({e})")

    print("\n--- Sample Candidate (Structured + Vector) ---")
    try:
        cursor.execute("""
            SELECT candidate_id, full_name, salary_expectation,
                   CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
            FROM candidate_pool
            FETCH FIRST 2 ROWS ONLY
        """)
        for row in cursor.fetchall():
            print(row)
    except Exception as e:
        print(f"Error fetching candidates: {e}")

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')
('CAND_001', 'Alex V.', 165000, '✅ Vector Present')

=== Verification Complete ===


In [45]:
import textwrap

def run_oracle_recruitment_scenario(role: str, width: int = 80):
    """
    Encapsulates Scenario A: Recruitment using the OracleRecruiter agent.

    Args:
        role (str): The job title or role description to search for.
        width (int): The max line length for the output display (default 80).

    Returns:
        tuple: (result, trace)
    """
    # 1. Define the simplified goal based only on the role
    recruitment_goal = role

    # 2. Visual separation for the start
    print("-" * width)
    print(f"Goal: {recruitment_goal}")
    print("-" * width)

    try:
        # 3. Execute the engine
        # Assuming run_universal_engine is defined in global scope
        result, trace = run_universal_engine(recruitment_goal)

        # 4. Present the text results nicely
        print("\n--- FINAL RECRUITMENT EMAILS ---\n")

        # Split by lines to preserve paragraph structure, then wrap lines individually
        if result:
            for line in str(result).splitlines():
                if line.strip(): # If line is not empty
                    print(textwrap.fill(line, width=width))
                else:
                    print() # Preserve empty lines
        else:
            print("No result returned.")

        print("-" * width)
        return result, trace

    except NameError:
        print("Error: `run_universal_engine` is not defined.")
        return None, None

In [46]:
#@title Scenario A : Searching for experienced Python developers
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience
--------------------------------------------------------------------------------

--- FINAL RECRUITMENT EMAILS ---

**Top Python Developer Candidates – Recruiter Summary**

---

### 1) Quinn R. (ID: CAND_005)

- **Total Experience:** 8 years
- **Current/Last Role:** Engineering Manager (previously hands-on technologist /
Python engineer)
- **Key Python Technologies:**
  - Core Python for backend and tooling
  - Likely experience with common Python ecosystems (e.g., web services,
scripting, automation) given career path
- **Notable Projects / Domains:**
  - Engineering leadership on Python-based projects
  - Team growth and mentorship around Python and related technologies
  - Aligning engineering delivery (including Python work) with business strategy
- **Overall Suitability (Senior/Experienced Roles):**
  - Strong fit for senior/lead Python roles th

In [47]:
#@ Scenario B Searching for experienced Python developers and writing an email
# The text will now automatically wrap at 80 characters for better readability
res, log = run_oracle_recruitment_scenario("Find Experienced Python Developers with experience and 250000 salary max and write a job interview email")

--------------------------------------------------------------------------------
Goal: Find Experienced Python Developers with experience and 250000 salary max and write a job interview email
--------------------------------------------------------------------------------

--- FINAL RECRUITMENT EMAILS ---

Subject: Interview Invitation – Experienced Python Developer Opportunity

Dear Quinn,

I hope you’re doing well. I’m reaching out because your strong Python background
and leadership experience appear to be an excellent match for our Experienced
Python Developer position.

This role focuses on designing and implementing robust, production-grade Python
services, collaborating closely with cross-functional teams, and contributing to
high-impact technical decisions. We’re looking for someone with substantial
hands-on Python experience who can bring both technical depth and a broader
engineering perspective.

The compensation for this role is competitive, with a maximum salary of 250,000

In [48]:
#@title Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
recruitment_goal = "Find Experienced Python Developers with a maxi salary of 200000 and experienced in Python services."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with a maxi salary of 200000 and experienced in Python services.

--- FINAL RECRUITMENT EMAILS ---
**Hiring Report: Python Services Candidates (< $200k)**

---

### Overview

This report summarizes and ranks candidates for a Python services–oriented role under $200k. The focus is on hands-on Python microservices experience, cloud-native architectures, and relevant backend skills. One candidate is a strong technical fit, one is more suited to a leadership/management role with less recent IC work, and one is not a Python-services match.

---

### Ranked Candidates

#### 1. Alex V. (CAND_001) – Primary Technical Fit

- **Experience level:**  
  - 12 years total software engineering experience

- **Python services skills:**  
  - Extensive work building scalable microservices in Python (and some Go)  
  - Deep experience with cloud-native architectures  
  - Led or executed migrations from legacy monoliths to microservices on AWS  
  - Strong CI/CD 

In [56]:
#@title Query powered by Oracle
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

recruitment_goal = "Find Experienced Python Developers with max salary 200000 and min 3 years experience. Then, write a personalized rejection email to the top candidate found, explicitly addressing them by their real name from the search results."
print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with max salary 200000 and min 3 years experience. Then, write a personalized rejection email to the top candidate found, explicitly addressing them by their real name from the search results.

--- FINAL RECRUITMENT EMAILS ---
Subject: Application Update

Hi Riley,

Thank you for taking the time to speak with us and for your interest in the role. We appreciate the thought and effort you put into the process, as well as your strong background in backend Java development and banking systems.

After careful consideration, we’ve decided to move forward with another candidate whose experience more closely matches our current needs. This was a difficult decision given the strength of your profile.

We truly appreciate your interest in our team and encourage you to consider applying again in the future if you see a role that aligns with your goals.

Wishing you all the best in your continued career,

[Your Name]  
[Your Title]  
[Company Name]


In [54]:
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
# UPGRADE: Explicitly used "140000" (integer format) to ensure Planner parsing accuracy.
recruitment_goal = "Find Experienced Python Developers with max salary 200000 and min 3 years experience, then write a personalized rejection email to the top candidate found, explicitly using their name and summarizing their profile."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

# --- RESULT HANDLING & DEBUGGING ---
if result:
    print("\n--- FINAL RECRUITMENT EMAILS ---")
    print(result)
else:
    print("\n⚠️ No Final Result returned by Engine. Inspecting Trace...")
    print(f"Trace Status: {trace.status}")

    # 1. Did we get a plan?
    if trace.plan:
        print(f"Plan Generated: {len(trace.plan)} steps")
        for step in trace.plan:
            print(f" - Step {step.get('step')}: {step.get('agent')} -> {step.get('input')}")
    else:
        print("❌ CRITICAL: No Plan was generated. Check your LLM / Planner logic.")

    # 2. Did we execute steps?
    if trace.steps:
        print(f"\nSteps Executed: {len(trace.steps)}")
        last_step = trace.steps[-1]
        print(f"Last Agent: {last_step['agent']}")
        print(f"Last Output Snippet: {str(last_step['output'])[:200]}...")
    else:
        print("❌ CRITICAL: Plan generated but no steps executed.")

Goal: Find Experienced Python Developers with max salary 200000 and min 3 years experience, then write a personalized rejection email to the top candidate found, explicitly using their name and summarizing their profile.

--- FINAL RECRUITMENT EMAILS ---
Hi Alex,

Thank you for taking the time to interview with us and for your interest in the role.

We were impressed by your 12 years of experience as a Senior Full Stack Engineer, especially your deep expertise in Python-based cloud‑native architectures and scalable microservices, as well as your leadership in migrating monolithic systems to AWS and implementing robust CI/CD and infrastructure-as-code practices.

After careful consideration, we’ve decided not to move forward with your application for this particular position. This decision was challenging and reflects the specific requirements and constraints of the role, not a lack of qualifications on your part.

With your permission, we would like to keep your profile on file and con

In [51]:
close=False
if close==True:
 OracleManager.close()